Q1. 학생 성적 보고서 생성기

In [6]:
# 표준 라이브러리 세 개
import csv # CSV 파일 읽기
import json # JSON 직렬화
import logging # 로그 출력

logging.basicConfig(level=logging.INFO) # INFO 이상 메시지 화면 출력 설정

# 함수 정의
def make_report(csv_path: str, json_path: str) -> int:
    try:
        with open(csv_path, "r", encoding="utf-8") as f: # CSV 파일 열기
            students = list(csv.DictReader(f)) # 각 행을 딕셔너리로 읽고 리스트로 저장
    # 파일이 존재하지 않을 시 WARNING 로고 및 0 반환
    except FileNotFoundError:
        logging.warning(f"CSV 파일이 없습니다: {csv_path}")
        return 0
    # 파일 인코딩이 UTF-8이 아닐 시 ERROR 로고 및 0 반환
    except UnicodeDecodeError:
        logging.error(f"CSV 파일 인코딩이 잘못되었습니다: {csv_path}")
        return 0

    report = [] # 리스트 생성

    for s in students: # CSV 딕셔너리 순회
        mid, fin, hw = s["중간"].strip(), s["기말"].strip(), s["과제"].strip()
        if any(v == "" for v in (mid, fin, hw)): # 세 점수 중 하나라도 빈 문자열이면 결측값
            # 값이 있는 점수는 int, 없는 점수는 None
            scores = {"중간": int(mid) if mid else None,
                      "기말": int(fin) if fin else None,
                      "과제": int(hw)  if hw  else None}
            avg, grade = None, None # 결측값이 있다면 둘 다 None
        else:
            mid, fin, hw = int(mid), int(fin), int(hw) # 세 점수가 모두 있으면 int로 반환
            avg    = mid * 0.3 + fin * 0.5 + hw * 0.2 # 가중 평균
            scores = {"중간": mid, "기말": fin, "과제": hw} # 딕셔너리 저장
            grade  = "A" if avg >= 90 else "B" if avg >= 80 else "C" if avg >= 70 else "F"

        logging.info(f'{s["이름"]}: 평균 {avg}, 등급 {grade}') # 학생 한 명을 처리할 때마다 이름, 평균, 등급을 INFO 로그로 출력
        report.append({"이름": s["이름"], "학번": s["학번"], "점수": scores, "평균": avg, "등급": grade}) # 한 학생의 결과를 딕셔너리로 만들고 리스트에 추가

    # 각 학생의 점수 딕셔너리를 json.dumps로 한 줄 문자열로 변환
    for st in report:
        st["점수"] = json.dumps(st["점수"], ensure_ascii=False)

    output = json.dumps(report, ensure_ascii=False, indent=2) # report 리스트를 JSON 문자열로 변환. 한글 유지. 들여쓰기 적용
    output = output.replace('"점수": "', '"점수": ').replace('\\"', '"').replace('}"', '}') # 따옴표, 이스케이프 제거

    # JSON 문자열 UTF-8로 파일에 저장
    with open(json_path, "w", encoding="utf-8") as f:
        f.write(output)

    # 학생 수 반환
    return len(report)

In [7]:
make_report("scores.csv", "report.json")

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 평균 None, 등급 None


4

설명: 
csv.DictReader로 CSV를 읽어 학생별로 가중 평균과 등급을 계산한 후 JSON으로 저장한다. 세 점수 중 하나라도 빈 칸이면 결측값으로 처리하여 평균, 등급을 None으로 출력하였고 값이 있는 점수는 개별 유지했다. CSV가 UTF-8 BOM 없음이기에 encoding="utf-8"을 사용했으며 ensure_ascii=False로 한글을 이스케이프 없이 그대로 보존했다. 예외는 FileNotFoundError(WARNING)와 UnicodeDecodeError(ERROR)로 구분해 기록하며 0을 반환하였다.

생성형 AI 참고 링크: https://claude.ai/share/cd21c23c-d64e-42d3-bd69-2dfa75466814

Q2. 사용자 정의 예외와 자모 분류

(a) 사용자 정의 예외 클래스 InvalidJamoError를 ValueError의 자식 클래스로 정의하라.

In [8]:
class InvalidJamoError(ValueError):
    pass

설명:
예를 들어, '가'는 타입(str)과 길이(1)는 올바르지만 값 자체가 부적절한 경우이므로 ValueError가 의미상 정확히 맞다. 또한, Exception과 달리 ValueError를 상속하면 except ValueError로도 잡을 수 있어 기존 코드와의 호환성도 유지된다.

(b) 함수 classify_jamo(c: str) -> str을 구현하라.

In [9]:
def classify_jamo(c: str) -> str:
    # 타입 검사: c가 str이 아니면 TypeError
    if not isinstance(c, str):
        raise TypeError(f"str 타입이 아닙니다: {c!r}")
    # 길이 검사: 문자열 길이가 1이 아니면 ValueError
    if len(c) != 1:
        raise ValueError(f"길이가 1이 아닙니다: {c!r}")
    
    # 유니코드 코드 포인트 추출
    code: int = ord(c)
    
    # 자음 판별
    if 0x3131 <= code <= 0x314E:
        return "자음"
    # 모음 판별
    elif 0x314F <= code <= 0x3163:
        return "모음"
    # 자음도 모음도 아니면 InvalidJamoError
    else:
        raise InvalidJamoError(f"한글 자모가 아닙니다: {c!r}")

설명:
조건 검사 순서를 타입, 길이, 유니코드 범위 순으로 배치하였는데, 문제 조건에서 길이 오류는 InvalidJamoError가 아님을 명시했기 때문에 길이 조건에서 InvalidJamoError가 아닌 일반 ValueError를 발생시켰다. 자모 판별은 힌트에 따라 ord(c)로 유니코드 코드 포인트를 얻어 자음(U+3131~U+314E), 모음(U+314F~U+3163) 범위를 직접 비교하였다.

(c) 다음 입력 리스트의 각 원소에 대해 classify_jamo를 호출하되, 예외가 발생하면 [예외종류] 메시지 형식으로 한 줄에 출력하고 다음 원소로 넘어가는 코드를 작성하라.

In [10]:
inputs = ["ㄱ", "ㅏ", "ㄲ", "가", "AB", 5, "ㅎ", "ㅣ", ""]

for item in inputs:
    try:
        # classify_jamo 호출문. 예외 없이 정상 실행 시 결과 출력
        result: str = classify_jamo(item)
        print(f"{item!r} → {result}")
    # str이 아닌 타입(ex) 5) 들어올 시 TypeError
    except TypeError as e:
        print(f"[TypeError] {e}")
    # 자모 아닌 문자(ex) "가") 들어올 시 InvalidJamoError
    except InvalidJamoError as e:
        print(f"[InvalidJamoError] {e}")
    # 길이가 1이 아닌 경우(ex) "AB", "") ValueError
    except ValueError as e:
        print(f"[ValueError] {e}")

'ㄱ' → 자음
'ㅏ' → 모음
'ㄲ' → 자음
[InvalidJamoError] 한글 자모가 아닙니다: '가'
[ValueError] 길이가 1이 아닙니다: 'AB'
[TypeError] str 타입이 아닙니다: 5
'ㅎ' → 자음
'ㅣ' → 모음
[ValueError] 길이가 1이 아닙니다: ''


설명:
for 문 안에서 try-except를 사용하여 예외가 발생해도 프로그램이 멈추지 않고 다음 원소로 넘어간다. except 순서는 TypeError, InvalidJamoError, ValueError 순으로 배치하였다. InvalidJamoError가 ValueError의 자식이므로 ValueError보다 먼저 써야 올바르게 잡히기 때문이다. 조건에 맞게 except Exception으로 싸잡아 잡지 않아 예상치 못한 버그가 숨지 않도록 하였다.

생성형 AI 참고 링크: https://claude.ai/share/72591499-8907-41d2-a7cd-316023d95f9c